# Classification Iris

## Prédiction de l'espèce d'une fleur Iris

Le but est de prédire l'espèce d'une fleur avec ses mesures.

### Librairies et données

In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
BASE_DIR = Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'streamlit_app' / 'artifacts'

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))
df = df.drop(columns='target')
df.head()

### 2. Préparation des données
La variable cible est `species`. La stratification conserve la même proportion de chaque espèce dans les jeux d'entraînement et de test.

In [ ]:
X = df.drop(columns='species')
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

### 3. Entraînement et comparaison
On teste plusieurs modèles et on compare leurs résultats.

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), {'model__C': [0.1, 1, 10]}),
    'K-Nearest Neighbors': (KNeighborsClassifier(), {'model__n_neighbors': [3, 5, 7], 'model__weights': ['uniform', 'distance']}),
    'Decision Tree': (DecisionTreeClassifier(random_state=RANDOM_STATE), {'model__max_depth': [None, 3, 5, 8], 'model__min_samples_leaf': [1, 2, 4]}),
    'Random Forest': (RandomForestClassifier(random_state=RANDOM_STATE), {'model__n_estimators': [100, 200], 'model__max_depth': [None, 3, 5], 'model__min_samples_leaf': [1, 2]})
}

fitted_models, rows = {}, []
for name, (model, params) in models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    grid = GridSearchCV(pipeline, params, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train, y_train)
    prediction = grid.predict(X_test)
    fitted_models[name] = grid.best_estimator_
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision': precision_score(y_test, prediction, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, prediction, average='weighted', zero_division=0),
        'F1-score': f1_score(y_test, prediction, average='weighted', zero_division=0),
        'Best parameters': grid.best_params_
    })

results = pd.DataFrame(rows).sort_values('F1-score', ascending=False).reset_index(drop=True)
results

### 4. Choix et sauvegarde du meilleur modèle
Seul le meilleur modèle est enregistré pour l'application Streamlit. Les autres modèles restent dans le notebook comme preuve de comparaison.

In [ ]:
best_model_name = results.loc[0, 'Model']
best_classifier = fitted_models[best_model_name]
joblib.dump(best_classifier, ARTIFACTS_DIR / 'best_classifier.joblib')
print(f'Modèle retenu : {best_model_name}')

### Conclusion
Le meilleur modèle est utilisé dans l'application Streamlit.